<a href="https://colab.research.google.com/github/madalamanikanta/ImageCaptioning_MiniProject/blob/manikanta-dev/02_Merge_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook 02 - Merge Datasets

## Step 1 - Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2 - Import Libraries

In [2]:
from pathlib import Path
import pandas as pd
import json
import os

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

## Step 3 - Define Project Paths

In [4]:
PROJECT_DIR = Path("/content/drive/MyDrive/ImageCaptioning_MiniProject")

DATASET_DIR = PROJECT_DIR / "DataSets"

EXTRACTED_DIR = DATASET_DIR / "Extracted"

PROCESSED_DIR = PROJECT_DIR / "Processed"

PROCESSED_DIR.mkdir(exist_ok=True)

print("Project :", PROJECT_DIR)
print("Extracted :", EXTRACTED_DIR)
print("Processed :", PROCESSED_DIR)

Project : /content/drive/MyDrive/ImageCaptioning_MiniProject
Extracted : /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted
Processed : /content/drive/MyDrive/ImageCaptioning_MiniProject/Processed


## Step 4 - Dataset Paths

In [5]:
FLICKR8K_DIR = EXTRACTED_DIR / "flickr8k"

FLICKR30K_DIR = (
    EXTRACTED_DIR
    / "flickr30k"
    / "flickr30k_images"
)

COCO_DIR = (
    EXTRACTED_DIR
    / "coco2017"
    / "coco2017"
)

print("Flickr8K :", FLICKR8K_DIR.exists())
print("Flickr30K:", FLICKR30K_DIR.exists())
print("COCO :", COCO_DIR.exists())

Flickr8K : True
Flickr30K: True
COCO : True


## Step 5 - Load Flickr8K Dataset

In [6]:
# Flickr8K Paths
flickr8k_images = FLICKR8K_DIR / "Images"
flickr8k_caption_file = FLICKR8K_DIR / "captions.txt"

# Load captions
flickr8k_df = pd.read_csv(flickr8k_caption_file)

# Create full image path
flickr8k_df["image_path"] = flickr8k_df["image"].apply(
    lambda x: str(flickr8k_images / x)
)

# Add dataset name
flickr8k_df["dataset"] = "Flickr8K"

# Keep only required columns
flickr8k_df = flickr8k_df[
    ["image_path", "caption", "dataset"]
]

print("Flickr8K Loaded Successfully!")
print()

print(flickr8k_df.head())

print()
print("Total Captions :", len(flickr8k_df))

Flickr8K Loaded Successfully!

                                                                                                        image_path  \
0  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   
1  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   
2  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   
3  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   
4  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   

                                                                    caption  \
0  A child in a pink dress is climbing up a set of stairs in an entry way .   
1                                     A girl going into a wooden building .   
2                      

## Step 6 - Load Flickr30K Dataset

In [7]:
# Flickr30K Paths
flickr30k_caption_file = (
    FLICKR30K_DIR /
    "results.csv"
)

flickr30k_images = (
    FLICKR30K_DIR /
    "flickr30k_images"
)

# Load captions
flickr30k_df = pd.read_csv(
    flickr30k_caption_file,
    delimiter="|"
)

# Remove unwanted spaces from column names
flickr30k_df.columns = [
    col.strip()
    for col in flickr30k_df.columns
]

# Keep only useful columns
flickr30k_df = flickr30k_df[
    ["image_name", "comment"]
]

# Create full image path
flickr30k_df["image_path"] = flickr30k_df["image_name"].apply(
    lambda x: str(flickr30k_images / x)
)

# Rename column
flickr30k_df.rename(
    columns={
        "comment": "caption"
    },
    inplace=True
)

# Add dataset name
flickr30k_df["dataset"] = "Flickr30K"

# Keep required columns
flickr30k_df = flickr30k_df[
    ["image_path", "caption", "dataset"]
]

print("Flickr30K Loaded Successfully!")
print()

print(flickr30k_df.head())

print()
print("Total Captions :", len(flickr30k_df))

Flickr30K Loaded Successfully!

                                                                                                                         image_path  \
0  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr30k/flickr30k_images/flickr30k_images/1000092795.jpg   
1  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr30k/flickr30k_images/flickr30k_images/1000092795.jpg   
2  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr30k/flickr30k_images/flickr30k_images/1000092795.jpg   
3  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr30k/flickr30k_images/flickr30k_images/1000092795.jpg   
4  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr30k/flickr30k_images/flickr30k_images/1000092795.jpg   

                                                                                caption  \
0   Two young guys with shaggy hair look at their hands while hang

## Step 7 - Load COCO 2017 Dataset

In [8]:
# COCO Paths
coco_train_images = COCO_DIR / "train2017"

coco_annotation_file = (
    COCO_DIR
    / "annotations"
    / "captions_train2017.json"
)

# Load JSON
with open(coco_annotation_file, "r") as f:
    coco_json = json.load(f)

# Image ID -> File Name Mapping
image_mapping = {}

for img in coco_json["images"]:
    image_mapping[img["id"]] = img["file_name"]

# Create DataFrame
rows = []

for ann in coco_json["annotations"]:

    image_name = image_mapping[ann["image_id"]]

    image_path = coco_train_images / image_name

    rows.append({
        "image_path": str(image_path),
        "caption": ann["caption"],
        "dataset": "COCO2017"
    })

coco_df = pd.DataFrame(rows)

print("COCO Loaded Successfully!")
print()

print(coco_df.head())

print()

print("Total Captions :", len(coco_df))

COCO Loaded Successfully!

                                                                                                           image_path  \
0  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000203564.jpg   
1  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000322141.jpg   
2  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000016977.jpg   
3  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000106140.jpg   
4  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/coco2017/coco2017/train2017/000000106140.jpg   

                                                               caption  \
0                   A bicycle replica with a clock as the front wheel.   
1                    A room with blue walls and a white sink and door.   
2  A car that seems to b

## Step 8 - Merge All Datasets

In [9]:
merged_df = pd.concat(
    [
        flickr8k_df,
        flickr30k_df,
        coco_df
    ],
    ignore_index=True
)

print("Datasets merged successfully!\n")

print(merged_df.head())

print()

print("Total Captions :", len(merged_df))

Datasets merged successfully!

                                                                                                        image_path  \
0  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   
1  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   
2  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   
3  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   
4  /content/drive/MyDrive/ImageCaptioning_MiniProject/DataSets/Extracted/flickr8k/Images/1000268201_693b08cb0e.jpg   

                                                                    caption  \
0  A child in a pink dress is climbing up a set of stairs in an entry way .   
1                                     A girl going into a wooden building .   
2                      

## Step 9 - Remove Duplicate Captions

In [10]:
before = len(merged_df)

merged_df.drop_duplicates(inplace=True)

after = len(merged_df)

print("Duplicates Removed :", before - after)
print("Remaining :", after)

Duplicates Removed : 186
Remaining : 790937


## Step 10 - Verify Image Paths

In [11]:
existing = merged_df["image_path"].apply(os.path.exists)

missing = (~existing).sum()

print("Missing Images :", missing)

merged_df = merged_df[existing].reset_index(drop=True)

print("Remaining Samples :", len(merged_df))

Missing Images : 191535
Remaining Samples : 599402


## Step 11 - Save Merged Dataset

In [12]:
output_file = PROCESSED_DIR / "merged_dataset.csv"

merged_df.to_csv(
    output_file,
    index=False
)

print("Merged dataset saved successfully!")
print(output_file)

Merged dataset saved successfully!
/content/drive/MyDrive/ImageCaptioning_MiniProject/Processed/merged_dataset.csv


## Step 12 - Dataset Statistics

In [13]:
print("=" * 50)
print("FINAL DATASET STATISTICS")
print("=" * 50)

print()

print(merged_df["dataset"].value_counts())

print()

print("Total Samples :", len(merged_df))

FINAL DATASET STATISTICS

dataset
COCO2017     400077
Flickr30K    158880
Flickr8K      40445
Name: count, dtype: int64

Total Samples : 599402
